In [1]:
import random
from collections import deque 

# Tạo class đại diện cho Node 
class Node:
    def __init__(self, state, parent, action, cost, name):
        self.state = state     
        self.parent = parent   
        self.action = action   
        self.cost = cost       
        self.name = name       

# Tạo trạng thái ban đầu (ngẫu nhiên)
def percept():
    list_num = list(range(9))
    random.shuffle(list_num)
    matrix = [list_num[i:i+3] for i in range(0, 9, 3)]
    return matrix

# Tìm vị trí của ô trống (số 0)
def interpret_input(matrix):
    for i in range(len(matrix)):
        for j in range(len(matrix)):
            if matrix[i][j] == 0:
                return (i, j)

# Tập luật (Trả về các hướng có thể đi)
def rules(state):
    x, y = state 
    actions = []
    if x > 0: actions.append("UP")
    if x < 2: actions.append("DOWN")
    if y > 0: actions.append("LEFT")
    if y < 2: actions.append("RIGHT")
    return actions

# Kiểm tra trạng thái đã hoàn thành hay chưa
def check_now(matrix):
    matrix_finish = [[1, 2, 3],
                     [4, 5, 6],
                     [7, 8, 0]]
    for i in range(len(matrix)):
        for j in range(len(matrix)):
            if matrix[i][j] != matrix_finish[i][j]:
                return 0
    return 1

# Thực hiện hành động 
def action(chosen, matrix, state):
    x, y = state 
    new_matrix = [row[:] for row in matrix] 
    value1 = new_matrix[x][y] 
    
    if chosen == "UP":
        value2 = new_matrix[x-1][y]
        new_matrix[x][y] = value2
        new_matrix[x-1][y] = value1
        return new_matrix, (x-1, y)
    elif chosen == "DOWN":
        value2 = new_matrix[x+1][y]
        new_matrix[x][y] = value2
        new_matrix[x+1][y] = value1
        return new_matrix, (x+1, y)
    elif chosen == "RIGHT":
        value2 = new_matrix[x][y+1]
        new_matrix[x][y] = value2
        new_matrix[x][y+1] = value1
        return new_matrix, (x, y+1)
    else: 
        value2 = new_matrix[x][y-1]
        new_matrix[x][y-1] = value1 
        new_matrix[x][y] = value2  
        return new_matrix, (x, y-1)

def format_state(state):
    return "\n".join(" ".join(str(val) for val in row) for row in state)

# In ra các action khi đã tới trạng thái đích
def print_path(goal_node_name, all_nodes):
    path = []
    current_name = goal_node_name
    
    while current_name is not None:
        node = all_nodes[current_name]
        path.append(node)
        current_name = node.parent
        
    path.reverse() 
    
    goal_node = all_nodes[goal_node_name]
    print(f"\n[THÀNH CÔNG] Đã tìm ra lời giải sau {goal_node.cost} bước đi!")
    for step_num, n in enumerate(path):
        action_name = n.action if n.action else "START"
        print(f"Bước {step_num} - Node: {n.name} \nState:\n{format_state(n.state)} \nHành động: {action_name}")
        print("-" * 20)

def main():
    matrix = percept()
    print("Trạng thái ban đầu:")
    for row in matrix:
        print(row)
        
    all_nodes = {}
    
    root_name = "A"
    root = Node(state=matrix, parent=None, action=None, cost=0, name=root_name)
    all_nodes[root_name] = root
    
    queue = deque([root]) 
    
    # SỬ DỤNG SET VÀ LƯU MA TRẬN DƯỚI DẠNG CHUỖI ĐỂ TỐI ƯU O(1)
    reached = set() 
    reached.add(str(root.state)) 
    
    limit = 100000
    step = 0
    
    while len(queue) > 0 and step < limit:
        
        current_node = queue.popleft() 
        
        if check_now(current_node.state) == 1:
            print_path(current_node.name, all_nodes)
            return 
        
        zero_pos = interpret_input(current_node.state)
        valid_actions = rules(zero_pos)
        
        child_index = 1
        for act in valid_actions:
            new_matrix, _ = action(act, current_node.state, zero_pos)
            new_matrix_str = str(new_matrix)
            
            if new_matrix_str not in reached:
                child_name = f"{current_node.name}{child_index}"
                
                child_node = Node(
                    state=new_matrix, 
                    parent=current_node.name, 
                    action=act, 
                    cost=current_node.cost + 1, 
                    name=child_name
                )
                
                queue.append(child_node)
                reached.add(new_matrix_str) 
                all_nodes[child_name] = child_node
                
                child_index += 1
            
        step += 1
        
    print(f"\nFailure: Đã duyệt {step} trạng thái nhưng không tìm thấy lời giải (Có thể bàn cờ sinh ra rơi vào trạng thái vô nghiệm).")

if __name__ == "__main__":
    main()

Trạng thái ban đầu:
[3, 8, 5]
[7, 6, 1]
[4, 0, 2]

Failure: Đã duyệt 100000 trạng thái nhưng không tìm thấy lời giải (Có thể bàn cờ sinh ra rơi vào trạng thái vô nghiệm).


In [2]:
%pip install pygame

^C
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pygame
import sys
import random
from collections import deque

#### LOGIC ####

class Node:
    def __init__(self, state, parent, action, cost, name):
        self.state = state
        self.parent = parent
        self.action = action
        self.cost = cost
        self.name = name

def percept():
    list_num = list(range(9))
    random.shuffle(list_num)
    return [list_num[i:i+3] for i in range(0, 9, 3)]

def interpret_input(matrix):
    for i in range(3):
        for j in range(3):
            if matrix[i][j] == 0:
                return (i, j)

def rules(state):
    x, y = state
    actions = []
    if x > 0: actions.append("UP")
    if x < 2: actions.append("DOWN")
    if y > 0: actions.append("LEFT")
    if y < 2: actions.append("RIGHT")
    return actions

def check_now(matrix):
    return 1 if matrix == [[1, 2, 3], [6, 5, 4], [7, 8, 0]] else 0

def action(chosen, matrix, state):
    x, y = state
    new_matrix = [row[:] for row in matrix]
    value1 = new_matrix[x][y]

    if chosen == "UP":
        new_matrix[x][y] = new_matrix[x-1][y]
        new_matrix[x-1][y] = value1
        return new_matrix, (x-1, y)
    elif chosen == "DOWN":
        new_matrix[x][y] = new_matrix[x+1][y]
        new_matrix[x+1][y] = value1
        return new_matrix, (x+1, y)
    elif chosen == "RIGHT":
        new_matrix[x][y] = new_matrix[x][y+1]
        new_matrix[x][y+1] = value1
        return new_matrix, (x, y+1)
    else:  # LEFT
        new_matrix[x][y] = new_matrix[x][y-1]
        new_matrix[x][y-1] = value1
        return new_matrix, (x, y-1)

def get_solution_path(goal_node_name, all_nodes):
    path = []
    current_name = goal_node_name
    while current_name is not None:
        node = all_nodes[current_name]
        path.append((node.state, node.action))
        current_name = node.parent
    path.reverse()
    return path  

# BFS1: Kiem tra goal khi lay node ra khoi hang cho 
def solve_8_puzzle_bfs1(initial_matrix):
    all_nodes = {}
    root_name = "A"
    root = Node(state=initial_matrix, parent=None, action=None, cost=0, name=root_name)
    all_nodes[root_name] = root
    queue = deque([root])
    reached = set()
    in_queue = set([str(root.state)])
    limit = 181440
    step = 0
    child_id = 1
    #check root
    if check_now(initial_matrix) == 1:
        return get_solution_path(root_name, all_nodes)
    

    while len(queue) > 0 and step < limit:
        current_node = queue.popleft()

        # Kiem tra goal khi lay ra khoi hang cho
        if check_now(current_node.state) == 1:
            return get_solution_path(current_node.name, all_nodes)
        else:
            reached.add(str(current_node.state))


        zero_pos = interpret_input(current_node.state)
        valid_actions = rules(zero_pos)

        for act in valid_actions:
            new_matrix, _ = action(act, current_node.state, zero_pos)
            new_matrix_str = str(new_matrix)

            if new_matrix_str not in reached and new_matrix_str not in in_queue:
                child_name = f"Node_{child_id}"
                child_node = Node(new_matrix, current_node.name, act, current_node.cost + 1, child_name)
                queue.append(child_node)
                in_queue.add(new_matrix_str)
                all_nodes[child_name] = child_node
                child_id += 1
        step += 1
    return None

# BFS2: Kiem tra goal ngay khi tao child 
def solve_8_puzzle_bfs2(initial_matrix):
    all_nodes = {}
    root_name = "A"
    root = Node(state=initial_matrix, parent=None, action=None, cost=0, name=root_name)
    all_nodes[root_name] = root

    # Kiem tra goal ngay tai trang thai dau
    if check_now(initial_matrix) == 1:
        return get_solution_path(root_name, all_nodes)

    queue = deque([root])
    reached = set()
    in_queue = set([str(root.state)])
    limit = 181440
    step = 0
    child_id = 1

    while len(queue) > 0 and step < limit:
        current_node = queue.popleft()

        zero_pos = interpret_input(current_node.state)
        valid_actions = rules(zero_pos)
        if check_now(current_node.state) == 1:
            return get_solution_path(current_node.name, all_nodes)
        else:
            reached.add(str(current_node.state))

        for act in valid_actions:
            new_matrix, _ = action(act, current_node.state, zero_pos)
            new_matrix_str = str(new_matrix)

            if new_matrix_str not in reached and new_matrix_str not in in_queue:
                child_name = f"Node_{child_id}"
                child_node = Node(new_matrix, current_node.name, act, current_node.cost + 1, child_name)

                # Kiem tra goal ngay khi tao child return luon neu thoa man
                if check_now(new_matrix) == 1:
                    all_nodes[child_name] = child_node
                    return get_solution_path(child_name, all_nodes)
                queue.append(child_node)
                in_queue.add(new_matrix_str)
                all_nodes[child_name] = child_node
                child_id += 1
           
        step += 1
    return None

### GIAO DIEN ###

WHITE      = (245, 245, 245)
BLACK      = (30,  30,  30)
DARK_BG    = (18,  18,  28)
PANEL_BG   = (26,  26,  40)
TILE_CLR   = (72, 130, 220)
TILE_EMPTY = (50,  50,  70)
GREEN      = (46, 204, 113)
GRAY       = (100, 100, 120)
ACCENT     = (255, 200,  80)
LOG_BG     = (22,  22,  36)
LOG_BORDER = (60,  60,  90)
LOG_HEAD   = (72, 130, 220)
LOG_EVEN   = (28,  28,  45)
LOG_ODD    = (32,  32,  50)
LOG_TEXT   = (200, 210, 255)
LOG_ACT    = (255, 200,  80)

ACTION_LABELS = {
    "UP":    "Di chuyen len",
    "DOWN":  "Di chuyen xuong",
    "LEFT":  "Di chuyen trai",
    "RIGHT": "Di chuyen phai",
    None:    "Trang thai ban dau",
}

def load_font(size, bold=False):
    candidates = [
        "segoeui", "arialuni", "notosans", "dejavusans",
        "freesans", "liberation sans", "tahoma",
    ]
    for name in candidates:
        try:
            f = pygame.font.SysFont(name, size, bold=bold)
            # Kiem tra co render duoc chu co dau khong
            test = f.render("Ti\u1ebfng Vi\u1ec7t", True, (255, 255, 255))
            if test.get_width() > 10:
                return f
        except Exception:
            pass
    return pygame.font.SysFont("arial", size, bold=bold)

def draw_board(screen, matrix, font, ox, oy, tile=90, margin=8):
    for i in range(3):
        for j in range(3):
            val = matrix[i][j]
            rx = ox + j * (tile + margin)
            ry = oy + i * (tile + margin)
            rect = pygame.Rect(rx, ry, tile, tile)

            if val == 0:
                pygame.draw.rect(screen, TILE_EMPTY, rect, border_radius=12)
                pygame.draw.rect(screen, GRAY, rect, 2, border_radius=12)
            else:
                pygame.draw.rect(screen, TILE_CLR, rect, border_radius=12)
                shadow = pygame.Rect(rx+3, ry+4, tile, tile)
                pygame.draw.rect(screen, (30, 60, 130), shadow, border_radius=12)
                pygame.draw.rect(screen, TILE_CLR, rect, border_radius=12)
                txt = font.render(str(val), True, WHITE)
                screen.blit(txt, txt.get_rect(center=rect.center))

def draw_button(screen, rect, label, font, active=False, hovered=False, disabled=False):
    if disabled:
        color = (50, 50, 70)
    elif active:
        color = GREEN
    elif hovered:
        color = (60, 110, 200)
    else:
        color = TILE_CLR
    pygame.draw.rect(screen, color, rect, border_radius=8)
    pygame.draw.rect(screen, WHITE if not disabled else GRAY, rect, 1, border_radius=8)
    txt = font.render(label, True, WHITE if not disabled else GRAY)
    screen.blit(txt, txt.get_rect(center=rect.center))

def draw_log_panel(screen, log_entries, scroll_offset, panel_rect, font_head, font_row):
    pygame.draw.rect(screen, LOG_BG, panel_rect, border_radius=10)
    pygame.draw.rect(screen, LOG_BORDER, panel_rect, 1, border_radius=10)

    px, py, pw, ph = panel_rect

    
    header_h = 32
    hdr_rect = pygame.Rect(px, py, pw, header_h)
    pygame.draw.rect(screen, LOG_HEAD, hdr_rect, border_radius=10)
    hdr_txt = font_head.render("Log cac buoc thuc hien", True, WHITE)
    screen.blit(hdr_txt, hdr_txt.get_rect(center=hdr_rect.center))

    # Hang
    row_h = 28
    content_area = pygame.Rect(px, py + header_h, pw, ph - header_h)
    clip = screen.get_clip()
    screen.set_clip(content_area)

    for idx, entry in enumerate(log_entries):
        row_y = py + header_h + idx * row_h - scroll_offset
        if row_y + row_h < py + header_h or row_y > py + ph:
            continue
        row_rect = pygame.Rect(px + 2, row_y, pw - 4, row_h - 2)
        bg = LOG_EVEN if idx % 2 == 0 else LOG_ODD

        # Highlight dong hien tai 
        if idx == len(log_entries) - 1:
            bg = (40, 70, 40)

        pygame.draw.rect(screen, bg, row_rect, border_radius=4)

        step_txt = font_row.render(f"B{entry['step']:03d}", True, ACCENT)
        screen.blit(step_txt, (px + 8, row_y + 5))

        act_txt = font_row.render(entry['action_label'], True, LOG_ACT)
        screen.blit(act_txt, (px + 58, row_y + 5))

        # In ma tran "1 2 3 | 4 5 6 | 7 8 0"
        rows_str = " | ".join(" ".join(str(v) for v in r) for r in entry['matrix'])
        mat_txt = font_row.render(rows_str, True, LOG_TEXT)
        screen.blit(mat_txt, (px + 210, row_y + 5))

    screen.set_clip(clip)

    # Thanh scroll
    total_h = len(log_entries) * row_h
    view_h = ph - header_h
    if total_h > view_h:
        ratio = view_h / total_h
        bar_h = max(20, int(view_h * ratio))
        bar_y = py + header_h + int(scroll_offset / total_h * view_h)
        pygame.draw.rect(screen, GRAY,
                         pygame.Rect(px + pw - 8, bar_y, 6, bar_h), border_radius=3)

def main_gui():
    pygame.init()
    W, H = 980, 500
    screen = pygame.display.set_mode((W, H))
    pygame.display.set_caption("8-Puzzle Solver")

    font_tile  = load_font(46, bold=True)
    font_med   = load_font(18, bold=True)
    font_small = load_font(15)
    font_log_h = load_font(15, bold=True)
    font_log_r = load_font(13)

    clock = pygame.time.Clock()

    current_matrix = percept()
    selected_algo  = "BFS1"
    status_msg     = "San sang! Nhan RUN de giai."

    is_animating   = False
    solution_path  = []   
    anim_index     = 0
    last_update    = 0
    DELAY          = 700  # ms moi buoc

    log_entries    = []
    log_scroll     = 0

    # Bo cuc
    BOARD_X, BOARD_Y = 30, 90
    CTRL_X           = 340
    LOG_RECT         = pygame.Rect(510, 10, 460, 480)

    btn_bfs1  = pygame.Rect(CTRL_X, 90,  150, 38)
    btn_bfs2  = pygame.Rect(CTRL_X, 140, 150, 38)
    btn_new   = pygame.Rect(CTRL_X, 200, 150, 38)
    btn_run   = pygame.Rect(CTRL_X, 260, 150, 50)

    running = True
    while running:
        screen.fill(DARK_BG)
        mouse = pygame.mouse.get_pos()

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False

            if event.type == pygame.MOUSEWHEEL:
                log_scroll = max(0, log_scroll - event.y * 20)

            if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1 and not is_animating:
                if btn_bfs1.collidepoint(event.pos):
                    selected_algo = "BFS1"

                elif btn_bfs2.collidepoint(event.pos):
                    selected_algo = "BFS2"

                elif btn_new.collidepoint(event.pos):
                    current_matrix = percept()
                    solution_path  = []
                    log_entries    = []
                    log_scroll     = 0
                    status_msg     = "Trang thai moi! Nhan RUN de giai."
                elif btn_run.collidepoint(event.pos):
                    log_entries = []
                    log_scroll  = 0
                    status_msg  = f"Dang tinh toan ({selected_algo})..."
                    draw_board(screen, current_matrix, font_tile, BOARD_X, BOARD_Y)
                    pygame.display.flip()

                    if selected_algo == "BFS1":
                        result = solve_8_puzzle_bfs1(current_matrix)
                    else:
                        result = solve_8_puzzle_bfs2(current_matrix)

                    if result:
                        solution_path = result
                        is_animating  = True
                        anim_index    = 0
                        last_update   = pygame.time.get_ticks()
                        status_msg    = f"Tim thay! {len(solution_path)-1} buoc ({selected_algo})"
                    else:
                        status_msg = "Khong tim thay duong di!"

        # hieu ung
        if is_animating:
            now = pygame.time.get_ticks()
            if now - last_update > DELAY:
                mat, act = solution_path[anim_index]
                current_matrix = mat
                label = ACTION_LABELS.get(act, act if act else "?")
                log_entries.append({
                    "step": anim_index,
                    "action_label": label,
                    "matrix": mat,
                })
                # Auto-scroll log xuong cuoi
                total_h = len(log_entries) * 28
                view_h  = LOG_RECT.height - 32
                if total_h > view_h:
                    log_scroll = total_h - view_h

                anim_index  += 1
                last_update  = now
                if anim_index >= len(solution_path):
                    is_animating = False
                    status_msg   = "HOAN THANH!"

        # --- Ve giao dien ---

        # Tieu de
        title = font_med.render("8-PUZZLE SOLVER", True, ACCENT)
        screen.blit(title, (BOARD_X, 50))

        # Board
        draw_board(screen, current_matrix, font_tile, BOARD_X, BOARD_Y)

        # Buttons
        draw_button(screen, btn_bfs1, "BFS 1 (late)",  font_small,
                    active=selected_algo=="BFS1",
                    hovered=btn_bfs1.collidepoint(mouse),
                    disabled=is_animating)
        draw_button(screen, btn_bfs2, "BFS 2 (early)", font_small,
                    active=selected_algo=="BFS2",
                    hovered=btn_bfs2.collidepoint(mouse),
                    disabled=is_animating)
        draw_button(screen, btn_new,  "Trang thai moi", font_small,
                    hovered=btn_new.collidepoint(mouse),
                    disabled=is_animating)
        draw_button(screen, btn_run,  "RUN", font_med,
                    hovered=btn_run.collidepoint(mouse),
                    disabled=is_animating)

        # Status
        status_color = GREEN if "HOAN THANH" in status_msg else (
                       (255, 80, 80) if "Khong" in status_msg else LOG_TEXT)
        stxt = font_small.render(status_msg, True, status_color)
        screen.blit(stxt, (CTRL_X, 330))

        # Log panel
        draw_log_panel(screen, log_entries, log_scroll, LOG_RECT, font_log_h, font_log_r)

        pygame.display.flip()
        clock.tick(60)

    pygame.quit()
    sys.exit()

if __name__ == "__main__":
    main_gui()

pygame 2.6.1 (SDL 2.28.4, Python 3.11.9)
Hello from the pygame community. https://www.pygame.org/contribute.html


SystemExit: 

C:\Users\TIN\AppData\Roaming\Python\Python311\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
